In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import cupy as cp
import cudf
from cuml.preprocessing import StandardScaler
from cuml.ensemble import RandomForestClassifier as cuRF
from cuml.linear_model import LogisticRegression as cuLogisticRegression
from cuml.metrics import accuracy_score

In [3]:
df = pd.read_csv("sample_to_run_info.csv")

/tmp/ipykernel_25496/2515147133.py:1: DtypeWarning: Columns (22,26,27,28,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("sample_to_run_info.csv")


In [4]:
thresh = int(len(df) * 0.5)          # minimum non-null count to keep a column
df = df.dropna(axis=1, thresh=thresh)

In [5]:
df = df.drop(['more', 'collection_date', 'longitude', 'latitude', 'sample_name', 'original_sample_description', 'sample_id', 'second_sample_id', 'disease', 'instrument_model'], axis=1)

In [6]:
# Check whether certain diseases are overrepresented on a given platform for instrument_model.
df

,project_id,run_id,experiment_type,nr_reads_sequenced,phenotype,country
0,PRJDB3418,DRR028772,AMPLICON,3000.0,Healthy,Japan
1,PRJDB3418,DRR028773,AMPLICON,3000.0,Healthy,Japan
2,PRJDB3418,DRR028774,AMPLICON,3000.0,Healthy,Japan
3,PRJDB3418,DRR028775,AMPLICON,3000.0,Healthy,Japan
4,PRJDB3418,DRR028776,AMPLICON,3000.0,Healthy,Japan
...,...,...,...,...,...,...
118860,PRJNA556801,SRR9851941,AMPLICON,51320.0,"Hepatitis, Autoimmune",China
118861,PRJNA556801,SRR9851942,AMPLICON,45711.0,Healthy,China
118862,PRJNA556801,SRR9851943,AMPLICON,60044.0,"Hepatitis, Autoimmune",China
118863,PRJNA556801,SRR9851944,AMPLICON,45185.0,"Hepatitis, Autoimmune",China


In [7]:
df['phenotype'].value_counts()

,count
phenotype,
Healthy,48241
Colorectal Neoplasms,5543
Crohn Disease,3516
COVID-19,2911
Parkinson Disease,2169
...,...
Colonic Polyps,5
Urolithiasis,5
Dry Eye Syndromes,4


In [8]:
# phenotype_counts = df['phenotype'].value_counts()
# valid_phenos = phenotype_counts[phenotype_counts >= 1000].index
# df = df[df['phenotype'].isin(valid_phenos)].copy()

keep_phenos = [
    "Healthy",
#     "Colorectal Neoplasms",
    "Crohn Disease",
    "Colitis, Ulcerative",
]

df = df[df["phenotype"].isin(keep_phenos)].copy()

In [9]:
df['country'].value_counts()

,count
country,
China,14709
United States of America,13830
Japan,5712
Denmark,2600
Netherlands,1521
Finland,1491
Spain,1301
Italy,1237
Germany,1121


In [10]:
df['experiment_type'].value_counts()

,count
experiment_type,
AMPLICON,37533
Metagenomics,16087


In [11]:
df

,project_id,run_id,experiment_type,nr_reads_sequenced,phenotype,country
0,PRJDB3418,DRR028772,AMPLICON,3000.0,Healthy,Japan
1,PRJDB3418,DRR028773,AMPLICON,3000.0,Healthy,Japan
2,PRJDB3418,DRR028774,AMPLICON,3000.0,Healthy,Japan
3,PRJDB3418,DRR028775,AMPLICON,3000.0,Healthy,Japan
4,PRJDB3418,DRR028776,AMPLICON,3000.0,Healthy,Japan
...,...,...,...,...,...,...
118848,PRJNA556801,SRR9851929,AMPLICON,42288.0,Healthy,China
118856,PRJNA556801,SRR9851937,AMPLICON,34177.0,Healthy,China
118857,PRJNA556801,SRR9851938,AMPLICON,39966.0,Healthy,China
118858,PRJNA556801,SRR9851939,AMPLICON,45325.0,Healthy,China


In [12]:
abundance = pd.read_csv('species_abundance.csv')

In [13]:
abundance

,id,loaded_uid,ncbi_taxon_id,taxon_rank_level,relative_abundance,accession_id
0,1,81104,-1,genus,1.951900,DRR358335
1,2,81104,544,genus,1.074570,DRR358335
2,3,81104,561,genus,0.849570,DRR358335
3,4,81104,570,genus,0.062180,DRR358335
4,5,81104,816,genus,23.296990,DRR358335
...,...,...,...,...,...,...
5541266,5541267,774,1980681,genus,0.286862,SRR9951896
5541267,5541268,774,2039302,genus,0.755403,SRR9951896
5541268,5541269,774,2316020,genus,1.415185,SRR9951896
5541269,5541270,774,2742598,genus,0.984892,SRR9951896


In [14]:
abundance['taxon_rank_level'].value_counts()

,count
taxon_rank_level,
genus,2780064
species,2761207


In [15]:
# Keep only genus level
abund_genus = abundance[abundance["taxon_rank_level"] == "genus"].copy()

# Drop rows with ncbi_taxon_id = -1 (unassigned/other)
abund_genus = abund_genus[abund_genus["ncbi_taxon_id"] != -1].copy()

In [16]:
# Pivot: run_id x ncbi_taxon_id, values = relative_abundance
abund_wide = (
    abund_genus
    .pivot_table(index="accession_id",      # this matches run_id in df
                 columns="ncbi_taxon_id",
                 values="relative_abundance",
                 aggfunc="sum",
                 fill_value=0.0)
    .reset_index()
)

# Make column names nicer (e.g., taxon_544, taxon_561, ...)
abund_wide.columns = [
    "run_id" if c == "accession_id" else f"taxon_{int(c)}"
    for c in abund_wide.columns
]

In [17]:
abundance

,id,loaded_uid,ncbi_taxon_id,taxon_rank_level,relative_abundance,accession_id
0,1,81104,-1,genus,1.951900,DRR358335
1,2,81104,544,genus,1.074570,DRR358335
2,3,81104,561,genus,0.849570,DRR358335
3,4,81104,570,genus,0.062180,DRR358335
4,5,81104,816,genus,23.296990,DRR358335
...,...,...,...,...,...,...
5541266,5541267,774,1980681,genus,0.286862,SRR9951896
5541267,5541268,774,2039302,genus,0.755403,SRR9951896
5541268,5541269,774,2316020,genus,1.415185,SRR9951896
5541269,5541270,774,2742598,genus,0.984892,SRR9951896


In [18]:
merged = df.merge(abund_wide, on="run_id", how="inner")

In [19]:
merged

,project_id,run_id,experiment_type,nr_reads_sequenced,phenotype,country,taxon_6,taxon_10,taxon_16,taxon_18,...,taxon_3090883,taxon_3109381,taxon_3119832,taxon_3120720,taxon_3120725,taxon_3120754,taxon_3121626,taxon_3277338,taxon_3373482,taxon_3409995
0,PRJDB3418,DRR028775,AMPLICON,3000.0,Healthy,Japan,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,PRJDB3418,DRR028783,AMPLICON,3000.0,Healthy,Japan,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,PRJDB3418,DRR028785,AMPLICON,3000.0,Healthy,Japan,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,PRJDB3418,DRR028787,AMPLICON,3000.0,Healthy,Japan,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,PRJDB3418,DRR028789,AMPLICON,3000.0,Healthy,Japan,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31201,PRJNA556801,SRR9851929,AMPLICON,42288.0,Healthy,China,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31202,PRJNA556801,SRR9851937,AMPLICON,34177.0,Healthy,China,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31203,PRJNA556801,SRR9851938,AMPLICON,39966.0,Healthy,China,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31204,PRJNA556801,SRR9851939,AMPLICON,45325.0,Healthy,China,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [20]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

In [21]:
keep_phenos = ["Healthy", "Crohn Disease", "Colitis, Ulcerative"]
df3 = merged[merged["phenotype"].isin(keep_phenos)].copy()

label_map = {
    "Healthy": 0,
    "Crohn Disease": 1,
    "Colitis, Ulcerative": 2,
}
df3["label"] = df3["phenotype"].map(label_map)

taxon_cols = [c for c in df3.columns if c.startswith("taxon_")]

gdf = cudf.from_pandas(df3[["label"] + taxon_cols])

X = gdf[taxon_cols]
y = gdf["label"]

In [22]:
from sklearn.model_selection import train_test_split

X_pd = df3[taxon_cols].values
y_pd = df3["label"].values

X_train_pd, X_test_pd, y_train_pd, y_test_pd = train_test_split(
    X_pd, y_pd, test_size=0.2, stratify=y_pd, random_state=42
)

X_train = cudf.DataFrame(X_train_pd, columns=taxon_cols)
X_test  = cudf.DataFrame(X_test_pd,  columns=taxon_cols)
y_train = cudf.Series(y_train_pd)
y_test  = cudf.Series(y_test_pd)

In [23]:
scaler = StandardScaler(with_mean=False)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

clf = cuLogisticRegression(
    penalty="l1",
    C=0.1,                 # tune for sparsity
    l1_ratio=None,         # pure L1
    fit_intercept=True,
    class_weight="balanced",
    max_iter=5000,
    solver="qn",           # cuML solver
    verbose=0
)

In [24]:
clf.fit(X_train_scaled, y_train)

LogisticRegression()

In [25]:
y_pred = clf.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print("GPU multinomial L1-logreg accuracy:", float(acc))

GPU multinomial L1-logreg accuracy: 0.9152515219480936


In [26]:
# coef_ is a cuDF DataFrame: shape (n_classes, n_features)
coefs_cudf = clf.coef_

# Convert to pandas / NumPy
coefs = coefs_cudf.to_pandas().to_numpy()   # (n_classes, n_features)

# classes_ is already a NumPy array (e.g., array([0, 1, 2]))
classes = clf.classes_

# Union of non-zero features across classes
nz_any = (coefs != 0).any(axis=0)
selected_taxa_union = np.array(taxon_cols)[nz_any]

print("Total taxa:", len(taxon_cols))
print("Selected by LASSO (any class):", len(selected_taxa_union))

selected_df = pd.DataFrame({
    "taxon": selected_taxa_union,
    "coef_l1_sum": np.abs(coefs[:, nz_any]).sum(axis=0)
}).sort_values("coef_l1_sum", ascending=False)

print(selected_df.head(30))


Total taxa: 2213
Selected by LASSO (any class): 626
             taxon  coef_l1_sum
137    taxon_35832     0.724913
352   taxon_574697     0.665977
297   taxon_292632     0.492688
549  taxon_2719313     0.488487
120    taxon_31977     0.470479
267   taxon_204475     0.433041
371   taxon_909656     0.421978
235   taxon_158846     0.421386
229   taxon_142577     0.418274
388  taxon_1283313     0.410338
598  taxon_2944148     0.381069
466  taxon_1906119     0.358997
485  taxon_1926677     0.349428
597  taxon_2944147     0.339322
510  taxon_2005473     0.327514
109    taxon_28209     0.319852
126    taxon_33024     0.313752
295   taxon_283168     0.308433
517  taxon_2048137     0.296697
70      taxon_1386     0.294705
379  taxon_1080709     0.286372
556  taxon_2764317     0.280789
248   taxon_172900     0.277263
220   taxon_119852     0.267111
609  taxon_2944199     0.266526
557  taxon_2767353     0.266393
316   taxon_379065     0.265409
321   taxon_400060     0.260027
48       taxon_872  

In [27]:
full_lasso = pd.concat(
    [df3.drop(columns=taxon_cols), df3[selected_taxa_union]],
    axis=1
)

In [28]:
full_lasso

,project_id,run_id,experiment_type,nr_reads_sequenced,phenotype,country,label,taxon_16,taxon_20,taxon_68,...,taxon_2976153,taxon_2981631,taxon_2983249,taxon_2983250,taxon_2983459,taxon_2983509,taxon_2994443,taxon_2995122,taxon_3025755,taxon_3028852
0,PRJDB3418,DRR028775,AMPLICON,3000.0,Healthy,Japan,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
1,PRJDB3418,DRR028783,AMPLICON,3000.0,Healthy,Japan,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
2,PRJDB3418,DRR028785,AMPLICON,3000.0,Healthy,Japan,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
3,PRJDB3418,DRR028787,AMPLICON,3000.0,Healthy,Japan,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
4,PRJDB3418,DRR028789,AMPLICON,3000.0,Healthy,Japan,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31201,PRJNA556801,SRR9851929,AMPLICON,42288.0,Healthy,China,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
31202,PRJNA556801,SRR9851937,AMPLICON,34177.0,Healthy,China,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.115789,0.0
31203,PRJNA556801,SRR9851938,AMPLICON,39966.0,Healthy,China,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.036941,0.0
31204,PRJNA556801,SRR9851939,AMPLICON,45325.0,Healthy,China,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0


In [29]:
print(full_lasso.shape)
print(full_lasso["phenotype"].value_counts())

(31206, 633)
phenotype
Healthy                28395
Crohn Disease           1748
Colitis, Ulcerative     1063
Name: count, dtype: int64


In [30]:
# full_lasso: pandas df with selected_taxa_union + 'label'
feature_cols = list(selected_taxa_union)

X = full_lasso[feature_cols].values
y = full_lasso["label"].values  # 0=Healthy,1=Crohn,2=UC

In [31]:
# Train/test split on CPU first
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Move to GPU (cuDF)
X_train = cudf.DataFrame(X_train_np, columns=feature_cols)
X_test  = cudf.DataFrame(X_test_np,  columns=feature_cols)
y_train = cudf.Series(y_train_np)
y_test  = cudf.Series(y_test_np)

In [32]:
# Move to GPU (cuDF)
X_train = cudf.DataFrame(X_train_np, columns=feature_cols)
X_test  = cudf.DataFrame(X_test_np,  columns=feature_cols)
y_train = cudf.Series(y_train_np)
y_test  = cudf.Series(y_test_np)

rf = cuRF(
    n_estimators=300,
    max_depth=20,       # <- set an explicit int, not None
    max_features=1.0,   # fraction of features; 1.0 ~ use all
    n_bins=16,
    bootstrap=True,
    random_state=42,
)

In [33]:
rf.fit(X_train, y_train)

RandomForestClassifier()

In [34]:
y_pred = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("GPU RandomForest accuracy:", float(acc))

GPU RandomForest accuracy: 0.9365587952579302


In [36]:
# Feature importances
importances = rf.feature_importances_
idx = importances.argsort()[::-1]

top_k = 30
top_feats = [(feature_cols[i], importances[i]) for i in idx[:top_k]]

top_importance_df = pd.DataFrame(top_feats, columns=["taxon", "rf_importance"])
print(top_importance_df)

            taxon  rf_importance
0   taxon_2719313       0.060153
1    taxon_909656       0.032156
2   taxon_2981631       0.029731
3    taxon_204475       0.029084
4     taxon_84111       0.027961
5      taxon_1678       0.024134
6    taxon_216851       0.023388
7       taxon_816       0.023148
8   taxon_2316020       0.021031
9      taxon_1301       0.016435
10   taxon_375288       0.014653
11  taxon_2815777       0.014000
12  taxon_2048137       0.013942
13   taxon_239759       0.013868
14    taxon_39948       0.012727
15  taxon_1506553       0.011900
16  taxon_2974251       0.011403
17      taxon_583       0.011334
18     taxon_1485       0.011181
19     taxon_1263       0.010358
20  taxon_2039302       0.010239
21   taxon_572511       0.010157
22   taxon_207244       0.009984
23    taxon_35832       0.009594
24      taxon_841       0.009296
25   taxon_283168       0.009203
26  taxon_2591381       0.009185
27     taxon_1730       0.008942
28   taxon_189330       0.008813
29    taxo

In [43]:
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score

In [39]:
# full_lasso: pandas df with selected_taxa_union and "label" (0=Healthy,1=Crohn,2=UC)
feature_cols = list(selected_taxa_union)

X = full_lasso[feature_cols].values
y = full_lasso["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest  = xgb.DMatrix(X_test,  label=y_test)

In [41]:
params = {
    "objective": "multi:softprob",
    "num_class": 3,
    "tree_method": "hist",   # GPU acceleration
    "predictor": "gpu_predictor",
    "max_depth": 6,
    "eta": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "eval_metric": "mlogloss",
    "device": "cuda",
}

bst = xgb.train(
    params,
    dtrain,
    num_boost_round=300,
    evals=[(dtrain, "train"), (dtest, "test")],
    early_stopping_rounds=30,
    verbose_eval=False,
)

/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [20:45:57] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "predictor" } are not used.

  self.starting_round = model.num_boosted_rounds()


In [44]:
y_proba = bst.predict(dtest)              # shape (n_samples, 3)
y_pred = np.argmax(y_proba, axis=1)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      1.00      0.98      5680
           1       0.85      0.54      0.66       350
           2       0.86      0.39      0.53       212

    accuracy                           0.95      6242
   macro avg       0.89      0.64      0.72      6242
weighted avg       0.95      0.95      0.94      6242



In [45]:
# Macro-averaged ROC-AUC (one-vs-rest)
y_test_oh = np.eye(3)[y_test]
auc = roc_auc_score(y_test_oh, y_proba, multi_class="ovr")
print("Macro ROC-AUC:", auc)

Macro ROC-AUC: 0.9583794090552149


In [46]:
# Feature importance
importance = bst.get_score(importance_type="gain")
# importance is a dict {"f0":gain,...} where f{i} maps to feature_cols[i]
items = []
for k, v in importance.items():
    idx = int(k[1:])  # drop 'f'
    items.append((feature_cols[idx], v))

In [47]:
imp_df = pd.DataFrame(items, columns=["taxon", "gain"]).sort_values("gain", ascending=False)
print(imp_df.head(30))

             taxon       gain
243  taxon_2126006  13.790490
263  taxon_2719313  12.986726
110   taxon_167375  12.714731
119   taxon_204475  11.709838
12       taxon_583   9.232702
20       taxon_836   9.068815
159   taxon_909656   8.239899
210  taxon_1918598   8.071318
288  taxon_2944152   7.517898
283  taxon_2944147   7.456381
89     taxon_85413   7.327545
198  taxon_1849822   7.234280
164  taxon_1080709   7.202523
102   taxon_142577   7.079005
35      taxon_1279   6.850068
71     taxon_40222   6.848148
237  taxon_2048137   6.177511
56     taxon_13687   5.993032
149   taxon_574697   5.816689
230  taxon_2005355   5.810105
172  taxon_1470349   5.776384
43      taxon_1578   5.338705
251  taxon_2316203   5.311986
258  taxon_2678884   5.027130
91     taxon_92793   5.009153
132   taxon_369926   4.910271
103   taxon_150022   4.880083
275  taxon_2815777   4.872920
33      taxon_1257   4.799126
69     taxon_35832   4.704521
